# QuantumEdge - Phase 3 Graphical Results and Compliance Review

Run after Notebooks 1 and 4. Missing compliance outputs are labelled rather than silently replaced.


## 1. Imports and folders

In [1]:
from pathlib import Path
import os

# Use local writable folders and a non-interactive plotting backend.
MPL_CACHE = Path(".matplotlib_cache")
MPL_CACHE.mkdir(parents=True, exist_ok=True)

os.environ["MPLCONFIGDIR"] = str(MPL_CACHE.resolve())
os.environ["MPLBACKEND"] = "Agg"

# Limit scientific-library threads in the qBraid environment.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

print("1. Importing standard libraries...")

import json
import math
import platform

print("2. Importing NumPy...")
import numpy as np
print("   NumPy loaded:", np.__version__)

print("3. Importing pandas...")
import pandas as pd
print("   pandas loaded:", pd.__version__)

print("4. Importing Matplotlib...")
import matplotlib

matplotlib.use("Agg", force=True)

import matplotlib.pyplot as plt
plt.ioff()

print("   Matplotlib loaded:", matplotlib.__version__)

print("5. Importing notebook display tools...")
from IPython.display import display, Markdown

RESULTS = Path("results")
FIGURES = Path("figures")
DASHBOARD = Path("dashboard_figures")

DASHBOARD.mkdir(parents=True, exist_ok=True)

print("\nImports completed successfully.")
print("Python:", platform.python_version())
print("Results folder:", RESULTS.resolve())
print("Dashboard folder:", DASHBOARD.resolve())

1. Importing standard libraries...
2. Importing NumPy...
   NumPy loaded: 2.5.1
3. Importing pandas...
   pandas loaded: 3.0.3
4. Importing Matplotlib...
   Matplotlib loaded: 3.11.0
5. Importing notebook display tools...

Imports completed successfully.
Python: 3.12.3
Results folder: /home/jovyan/phase 3 third run/results
Dashboard folder: /home/jovyan/phase 3 third run/dashboard_figures


In [2]:
import matplotlib.pyplot as plt

plt.close("all")

## 2. Provenance-aware loaders

In [3]:
# ============================================================
# PROVENANCE-AWARE RESULT LOADING
# ============================================================

REFERENCE_HEADLINE = pd.DataFrame([
    [
        "QRC Dual+Pauli+FB (calibrated)",
        7.532e-5,
        0.3261,
        1.465,
        0.571,
        0.475,
    ],
    [
        "QRC Dual+Pauli+FB",
        7.725e-5,
        0.3794,
        1.298,
        0.617,
        0.441,
    ],
    [
        "ESN",
        7.993e-5,
        0.4588,
        1.101,
        0.591,
        0.261,
    ],
    [
        "HAR-RV",
        8.005e-5,
        0.4129,
        1.262,
        0.593,
        0.302,
    ],
    [
        "LSTM (early-stopped)",
        8.231e-5,
        0.4297,
        1.630,
        0.576,
        0.499,
    ],
    [
        "Persistence",
        9.530e-5,
        0.6707,
        0.453,
        0.577,
        0.041,
    ],
    [
        "GARCH(1,1)",
        1.010e-4,
        0.6656,
        -3.098,
        0.451,
        0.329,
    ],
], columns=[
    "Model",
    "RMSE",
    "QLIKE",
    "MZ slope",
    "Regime",
    "Sharpe",
]).set_index("Model")


REFERENCE_OXFORD = pd.DataFrame([
    [
        "QRC Dual+Pauli+FB",
        5.594e-5,
        0.2425,
        1.274,
    ],
    [
        "HAR-RV",
        5.599e-5,
        0.2351,
        1.316,
    ],
    [
        "Persistence",
        6.067e-5,
        0.2612,
        0.672,
    ],
], columns=[
    "Model",
    "RMSE",
    "QLIKE",
    "MZ slope",
])


# ============================================================
# PROVENANCE RECORD
# ============================================================

provenance = []


def add_provenance(
    evidence,
    source,
    path,
    note="",
):
    provenance.append({
        "Evidence": evidence,
        "Source": source,
        "Path": str(path),
        "Note": note,
    })


def load_csv(
    path,
    *,
    index_col=None,
    fallback=None,
    label=None,
):
    path = Path(path)
    evidence_name = label or path.name

    if path.exists():
        frame = pd.read_csv(
            path,
            index_col=index_col,
        )

        add_provenance(
            evidence=evidence_name,
            source="submitted generated file",
            path=path,
            note="Generated by Notebook 1.",
        )

        return frame

    if fallback is not None:
        add_provenance(
            evidence=evidence_name,
            source="LOCKED REFERENCE FALLBACK",
            path="not found",
            note=(
                "Generated result file was absent; "
                "locked reference values are displayed."
            ),
        )

        return fallback.copy()

    add_provenance(
        evidence=evidence_name,
        source="not available",
        path="not found",
        note="No generated file or reference fallback was available.",
    )

    return None


# ============================================================
# SIMULATOR AND AUTHENTIC-RV RESULTS
# ============================================================

headline = load_csv(
    RESULTS / "headline_metrics.csv",
    index_col=0,
    fallback=REFERENCE_HEADLINE,
    label="Headline forecast metrics",
)

oxford = load_csv(
    RESULTS / "oxford_man_metrics.csv",
    fallback=REFERENCE_OXFORD,
    label="Oxford-Man authentic-RV metrics",
)

ablation = load_csv(
    RESULTS / "ablation.csv",
    index_col=0,
    label="Ablation",
)

scaling = load_csv(
    RESULTS / "reservoir_scaling.csv",
    label="Reservoir-size scaling",
)

encoding = load_csv(
    RESULTS / "encoding_density.csv",
    label="Encoding density",
)

shots = load_csv(
    RESULTS / "shot_budget.csv",
    label="Shot budget",
)

noise = load_csv(
    RESULTS / "noise_zne.csv",
    label="Noise and ZNE",
)


# ============================================================
# RECOVERED IBM JOB EVIDENCE
# ============================================================

recovered_metadata_files = sorted(
    RESULTS.glob(
        "recovered_ibm_*_metadata.json"
    )
)

recovered_evs_files = sorted(
    RESULTS.glob(
        "recovered_ibm_*_evs.csv"
    )
)

recovered_hardware_metadata = []
recovered_hardware_values = None


# Load recovered IBM metadata.
for metadata_file in recovered_metadata_files:
    try:
        metadata = json.loads(
            metadata_file.read_text(
                encoding="utf-8"
            )
        )

        recovered_hardware_metadata.append(
            metadata
        )

    except Exception as exc:
        print(
            "Warning: could not read",
            metadata_file,
            ":",
            exc,
        )


if recovered_hardware_metadata:
    job_ids = [
        str(item.get("job_id"))
        for item in recovered_hardware_metadata
        if item.get("job_id")
    ]

    add_provenance(
        evidence="IBM recovered job metadata",
        source="submitted recovered IBM job evidence",
        path="; ".join(
            str(path)
            for path in recovered_metadata_files
        ),
        note=(
            "Completed IBM workload retrieved directly "
            f"from IBM. Job ID(s): {', '.join(job_ids)}."
        ),
    )


# Load recovered expectation values.
recovered_frames = []

for result_file in recovered_evs_files:
    try:
        recovered_frame = pd.read_csv(
            result_file
        )

        recovered_frame["source_file"] = (
            result_file.name
        )

        recovered_frames.append(
            recovered_frame
        )

    except Exception as exc:
        print(
            "Warning: could not read",
            result_file,
            ":",
            exc,
        )


if recovered_frames:
    recovered_hardware_values = pd.concat(
        recovered_frames,
        ignore_index=True,
    )

    add_provenance(
        evidence="IBM recovered expectation values",
        source="submitted recovered IBM job evidence",
        path="; ".join(
            str(path)
            for path in recovered_evs_files
        ),
        note=(
            f"{len(recovered_hardware_values)} expectation "
            "values recovered from completed IBM job(s)."
        ),
    )


# ============================================================
# COMPLETE IBM HARDWARE SUMMARY
# ============================================================

hardware_summary = None
hardware_obs = None

hardware_path = (
    RESULTS / "hardware_validation.json"
)

hardware_observables_path = (
    RESULTS / "hardware_observables.csv"
)


if hardware_path.exists():
    hardware_summary = json.loads(
        hardware_path.read_text(
            encoding="utf-8"
        )
    )

    add_provenance(
        evidence="IBM hardware summary",
        source="submitted generated file",
        path=hardware_path,
        note=(
            "Complete hardware summary generated by "
            "the reproducibility notebook."
        ),
    )

else:
    # Locked metrics from the prior successful
    # three-window ibm_fez validation.
    hardware_summary = {
        "backend": "ibm_fez",
        "qubits": 7,
        "transpiled_depth": 68,
        "shots_per_circuit": 4096,
        "windows": 3,
        "wall_seconds": 211,
        "observables": 39,
        "feature_correlation": 0.989,
        "feature_mae": 0.0569,
        "_reference_fallback": True,
        "_supported_by_recovered_job": bool(
            recovered_hardware_metadata
            or recovered_frames
        ),
    }

    if (
        recovered_hardware_metadata
        or recovered_frames
    ):
        hardware_note = (
            "Correlation and MAE are locked reference "
            "results from the prior successful three-window "
            "validation. The package also contains recovered "
            "expectation values and metadata for a completed "
            "IBM job, but not the complete simulator/hardware "
            "pairing needed to recalculate the three-window metrics."
        )
    else:
        hardware_note = (
            "Correlation and MAE are locked reference "
            "results. No recovered IBM job evidence was found."
        )

    add_provenance(
        evidence="IBM hardware summary",
        source="LOCKED REFERENCE FALLBACK",
        path="not found",
        note=hardware_note,
    )


# Load complete simulator-versus-hardware pairs only
# when the correct file exists.
if hardware_observables_path.exists():
    hardware_obs = pd.read_csv(
        hardware_observables_path
    )

    add_provenance(
        evidence=(
            "IBM complete simulator/hardware "
            "observable pairs"
        ),
        source="submitted generated file",
        path=hardware_observables_path,
        note=(
            "Contains matched simulator and IBM "
            "expectation values used to calculate "
            "feature correlation and MAE."
        ),
    )

else:
    hardware_obs = None

    add_provenance(
        evidence=(
            "IBM complete simulator/hardware "
            "observable pairs"
        ),
        source="not available",
        path="not found",
        note=(
            "Recovered IBM expectation values do not "
            "contain the complete matching simulator "
            "values for all three validation windows."
        ),
    )


# ============================================================
# DISPLAY PROVENANCE
# ============================================================

provenance_table = pd.DataFrame(
    provenance
)

display(
    provenance_table.style.set_properties(
        **{
            "text-align": "left",
            "white-space": "normal",
        }
    )
)


if (
    provenance_table["Source"]
    == "LOCKED REFERENCE FALLBACK"
).any():
    display(Markdown(
        "> **Provenance notice:** At least one displayed "
        "result uses a locked reference value. Recovered IBM "
        "job evidence is identified separately and is not "
        "presented as a newly generated three-window summary."
    ))


if recovered_hardware_metadata:
    print("\nRecovered IBM job metadata:")

    for item in recovered_hardware_metadata:
        print(
            "- Job:",
            item.get("job_id"),
            "| Status:",
            item.get("status"),
            "| Backend:",
            item.get("backend"),
        )


if recovered_hardware_values is not None:
    print(
        "\nRecovered IBM expectation values:",
        len(recovered_hardware_values),
    )

    display(
        recovered_hardware_values.head()
    )

,Evidence,Source,Path,Note
0,Headline forecast metrics,submitted generated file,results/headline_metrics.csv,Generated by Notebook 1.
1,Oxford-Man authentic-RV metrics,submitted generated file,results/oxford_man_metrics.csv,Generated by Notebook 1.
2,Ablation,submitted generated file,results/ablation.csv,Generated by Notebook 1.
3,Reservoir-size scaling,submitted generated file,results/reservoir_scaling.csv,Generated by Notebook 1.
4,Encoding density,submitted generated file,results/encoding_density.csv,Generated by Notebook 1.
5,Shot budget,submitted generated file,results/shot_budget.csv,Generated by Notebook 1.
6,Noise and ZNE,submitted generated file,results/noise_zne.csv,Generated by Notebook 1.
7,IBM hardware summary,submitted generated file,results/hardware_validation.json,Complete hardware summary generated by the reproducibility notebook.
8,IBM complete simulator/hardware observable pairs,submitted generated file,results/hardware_observables.csv,Contains matched simulator and IBM expectation values used to calculate feature correlation and MAE.


In [4]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

display(
    provenance_table.style
    .set_properties(
        subset=["Evidence"],
        **{
            "text-align": "left",
            "min-width": "230px",
            "white-space": "normal",
        }
    )
    .set_properties(
        subset=["Source"],
        **{
            "text-align": "left",
            "min-width": "220px",
            "white-space": "normal",
        }
    )
    .set_properties(
        subset=["Path"],
        **{
            "text-align": "left",
            "min-width": "260px",
            "white-space": "normal",
        }
    )
    .set_properties(
        subset=["Note"],
        **{
            "text-align": "left",
            "min-width": "420px",
            "white-space": "normal",
        }
    )
)

,Evidence,Source,Path,Note
0,Headline forecast metrics,submitted generated file,results/headline_metrics.csv,Generated by Notebook 1.
1,Oxford-Man authentic-RV metrics,submitted generated file,results/oxford_man_metrics.csv,Generated by Notebook 1.
2,Ablation,submitted generated file,results/ablation.csv,Generated by Notebook 1.
3,Reservoir-size scaling,submitted generated file,results/reservoir_scaling.csv,Generated by Notebook 1.
4,Encoding density,submitted generated file,results/encoding_density.csv,Generated by Notebook 1.
5,Shot budget,submitted generated file,results/shot_budget.csv,Generated by Notebook 1.
6,Noise and ZNE,submitted generated file,results/noise_zne.csv,Generated by Notebook 1.
7,IBM hardware summary,submitted generated file,results/hardware_validation.json,Complete hardware summary generated by the reproducibility notebook.
8,IBM complete simulator/hardware observable pairs,submitted generated file,results/hardware_observables.csv,Contains matched simulator and IBM expectation values used to calculate feature correlation and MAE.


## 3. Headline forecast performance

In [5]:
# Normalize column names that may arrive with different capitalization.
headline.columns = [str(c).strip() for c in headline.columns]
display(headline)

plot_order = headline.sort_values("RMSE", ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(plot_order.index, plot_order["RMSE"] * 1e5)
ax.set_title("One-day-ahead realized-variance forecast — RMSE")
ax.set_xlabel("RMSE (×10⁻⁵; lower is better)")
for bar, value in zip(bars, plot_order["RMSE"] * 1e5):
    ax.text(value, bar.get_y() + bar.get_height()/2, f" {value:.3f}", va="center", fontsize=8)
fig.tight_layout()
fig.savefig(DASHBOARD / "01_headline_rmse.png", dpi=180)
plt.show()

qlike_order = headline.sort_values("QLIKE", ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(qlike_order.index, qlike_order["QLIKE"])
ax.set_title("One-day-ahead realized-variance forecast — QLIKE")
ax.set_xlabel("QLIKE (lower is better)")
for bar, value in zip(bars, qlike_order["QLIKE"]):
    ax.text(value, bar.get_y() + bar.get_height()/2, f" {value:.4f}", va="center", fontsize=8)
fig.tight_layout()
fig.savefig(DASHBOARD / "02_headline_qlike.png", dpi=180)
plt.show()

qrc = headline.loc["QRC Dual+Pauli+FB (calibrated)"]
har = headline.loc["HAR-RV"]
improvement = pd.DataFrame({
    "Metric": ["RMSE reduction vs HAR-RV", "QLIKE reduction vs HAR-RV"],
    "Improvement (%)": [
        100 * (har["RMSE"] - qrc["RMSE"]) / har["RMSE"],
        100 * (har["QLIKE"] - qrc["QLIKE"]) / har["QLIKE"],
    ],
})
display(improvement.style.format({"Improvement (%)": "{:.1f}%"}))

,Source,RMSE,QLIKE,MZ_slope,MZ_p,Regime_3state,Sharpe
QRC Dual+Pauli+FB (calibrated),QRC,0.000075,0.326062,1.465467,2.488483e-11,0.570667,0.475253
QRC Dual+Pauli+FB,QRC,0.000077,0.379357,1.297628,1.543359e-19,0.617333,0.440522
ESN,Classical,0.000080,0.458802,1.100550,3.698216e-14,0.590667,0.261033
HAR-RV,Classical,0.000080,0.412880,1.262071,8.137178e-15,0.593333,0.301937
LSTM,Classical,0.000082,0.429712,1.629710,2.079817e-28,0.576000,0.498900
Persistence,Classical,0.000095,0.670659,0.452926,1.267065e-52,0.577333,0.041012
"GARCH(1,1)",Classical,0.000101,0.665637,-3.097959,3.302046e-46,0.450667,0.329065


,Metric,Improvement (%)
0,RMSE reduction vs HAR-RV,5.9%
1,QLIKE reduction vs HAR-RV,21.0%


**Interpretation.** The calibrated QRC is the strongest headline model on RMSE and QLIKE. The Mincer–Zarnowitz slope remains above one, so residual bias is reported rather than hidden. Sharpe is secondary and is not used to claim dominance.

## 4. Calm-versus-turbulent regime detection

In [6]:
regime = pd.DataFrame({
    "Method": ["QRC-feature classifier", "Majority baseline"],
    "Accuracy": [0.781, 0.647],
})
fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(regime["Method"], regime["Accuracy"] * 100)
ax.set_ylim(0, 100)
ax.set_ylabel("Accuracy (%)")
ax.set_title("Two-state volatility regime detection")
for bar, value in zip(bars, regime["Accuracy"] * 100):
    ax.text(bar.get_x() + bar.get_width()/2, value, f"{value:.1f}%", ha="center", va="bottom")
fig.tight_layout()
fig.savefig(DASHBOARD / "03_regime_detection.png", dpi=180)
plt.show()
display(regime.style.format({"Accuracy": "{:.1%}"}))

,Method,Accuracy
0,QRC-feature classifier,78.1%
1,Majority baseline,64.7%


## 5. Authentic Oxford-Man five-minute realized variance

In [7]:
oxford.columns = [str(c).strip() for c in oxford.columns]
display(oxford)

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(oxford["Model"], oxford["RMSE"] * 1e5)
ax.set_ylabel("RMSE (×10⁻⁵)")
ax.set_title("Authentic five-minute realized variance — RMSE")
for bar, value in zip(bars, oxford["RMSE"] * 1e5):
    ax.text(bar.get_x() + bar.get_width()/2, value, f"{value:.3f}", ha="center", va="bottom", fontsize=8)
fig.tight_layout()
fig.savefig(DASHBOARD / "04_oxford_rmse.png", dpi=180)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(oxford["Model"], oxford["QLIKE"])
ax.set_ylabel("QLIKE")
ax.set_title("Authentic five-minute realized variance — QLIKE")
for bar, value in zip(bars, oxford["QLIKE"]):
    ax.text(bar.get_x() + bar.get_width()/2, value, f"{value:.4f}", ha="center", va="bottom", fontsize=8)
fig.tight_layout()
fig.savefig(DASHBOARD / "05_oxford_qlike.png", dpi=180)
plt.show()

,Model,RMSE,QLIKE,MZ_slope
0,QRC Dual+Pauli+FB,0.000056,0.242511,1.273768
1,HAR-RV,0.000056,0.235088,1.316347
2,Persistence,0.000061,0.261231,0.671976


**Interpretation.** On genuine five-minute realized variance, QRC and HAR-RV are effectively tied. The correct claim is **competitive with the strongest classical model**, not statistically superior. A Diebold–Mariano test remains future work.

## 6. Component ablation

In [8]:
if ablation is None:
    display(Markdown("> Ablation CSV is not present. Run the reproducibility notebook to generate `results/ablation.csv`."))
else:
    ablation.columns = [str(c).strip() for c in ablation.columns]
    display(ablation)
    view = ablation.sort_values("RMSE", ascending=False)
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.barh(view.index, view["RMSE"] * 1e5)
    ax.set_xlabel("RMSE (×10⁻⁵; lower is better)")
    ax.set_title("QRC architecture ablation")
    for bar, value in zip(bars, view["RMSE"] * 1e5):
        ax.text(value, bar.get_y() + bar.get_height()/2, f" {value:.3f}", va="center", fontsize=8)
    fig.tight_layout()
    fig.savefig(DASHBOARD / "06_ablation.png", dpi=180)
    plt.show()

,RMSE,QLIKE
Dual Pauli +FB,0.000077,0.379357
Single +Ent -FB,0.000077,0.401962
Single Pauli -FB,0.000078,0.402615
Dual +Ent -FB,0.000078,0.406956
Dual +Ent +FB,0.000078,0.393622
Single +Ent +FB,0.000079,0.409723


## 7. Reservoir-size and encoding-density robustness

In [9]:
if scaling is None:
    display(Markdown("> Scaling CSV is not present."))
else:
    display(scaling)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for depth in sorted(scaling["p"].unique()):
        subset = scaling[scaling["p"] == depth].sort_values("n")
        ax.plot(subset["n"], subset["RMSE"] * 1e5, marker="o", label=f"p={depth}")
    ax.set_xlabel("Reservoir qubits")
    ax.set_ylabel("RMSE (×10⁻⁵)")
    ax.set_title("Reservoir-size scaling")
    ax.legend()
    fig.tight_layout()
    fig.savefig(DASHBOARD / "07_scaling.png", dpi=180)
    plt.show()

if encoding is None:
    display(Markdown("> Encoding-density CSV is not present."))
else:
    display(encoding)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(encoding["encoding_fraction"], encoding["RMSE"] * 1e5, marker="o")
    ax.set_xlabel("Encoded fraction of reservoir")
    ax.set_ylabel("RMSE (×10⁻⁵)")
    ax.set_title("Encoding-density robustness")
    fig.tight_layout()
    fig.savefig(DASHBOARD / "08_encoding_density.png", dpi=180)
    plt.show()

,n,p,RMSE,QLIKE,seconds
0,5,2,0.000079,0.403568,7.4
1,5,4,0.000084,0.457577,11.2
2,7,2,0.000080,0.404380,10.4
3,7,4,0.000080,0.434257,16.1
4,9,2,0.000078,0.405327,13.6
5,9,4,0.000079,0.402955,21.8
6,11,2,0.000079,0.412659,19.0
7,11,4,0.000078,0.412439,31.5


,encoding_fraction,qubits_encoded,RMSE,QLIKE
0,0.34,3,0.000080,0.402906
1,0.67,6,0.000079,0.402758
2,1.00,9,0.000078,0.405327


## 8. Finite-shot robustness

In [10]:
if shots is None:
    display(Markdown("> Shot-budget CSV is not present."))
else:
    display(shots)
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot([str(v) for v in shots["shots"]], shots["RMSE"] * 1e5, marker="o")
    ax.set_xlabel("Shots")
    ax.set_ylabel("RMSE (×10⁻⁵)")
    ax.set_title("Shot-budget convergence")
    fig.tight_layout()
    fig.savefig(DASHBOARD / "09_shot_budget.png", dpi=180)
    plt.show()

,shots,RMSE,QLIKE
0,256,0.000080,0.426551
1,1024,0.000080,0.429308
2,4096,0.000079,0.415042
3,exact,0.000078,0.405327


## 9. Noise and zero-noise extrapolation

In [11]:
if noise is None:
    display(Markdown("> Noise/ZNE CSV is not present."))
else:
    display(noise)
    positions = np.arange(len(noise))
    width = 0.38
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(positions - width/2, noise["raw_mae"], width, label="Raw noisy")
    ax.bar(positions + width/2, noise["zne_mae"], width, label="ZNE mitigated")
    ax.set_xticks(positions)
    ax.set_xticklabels(noise["depol_1q"])
    ax.set_xlabel("One-qubit depolarizing rate")
    ax.set_ylabel("Feature MAE versus ideal")
    ax.set_title("Noise impact and zero-noise extrapolation")
    ax.legend()
    fig.tight_layout()
    fig.savefig(DASHBOARD / "10_noise_zne.png", dpi=180)
    plt.show()

,depol_1q,depol_2q,raw_mae,zne_mae,recovered_percent
0,0.005,0.01,0.026138,0.005283,80
1,0.010,0.02,0.050611,0.018180,64
2,0.020,0.04,0.094968,0.054900,42


## 10. IBM hardware feature agreement

In [12]:
summary_frame = pd.DataFrame({
    "Measure": [
        "Backend", "Qubits", "Transpiled depth", "Shots/circuit",
        "Validation windows", "Observables", "Feature correlation", "Feature MAE"
    ],
    "Value": [
        hardware_summary.get("backend"),
        hardware_summary.get("qubits"),
        hardware_summary.get("transpiled_depth"),
        hardware_summary.get("shots_per_circuit"),
        hardware_summary.get("windows"),
        hardware_summary.get("observables"),
        hardware_summary.get("feature_correlation"),
        hardware_summary.get("feature_mae"),
    ],
})
display(summary_frame)

if hardware_summary.get("_reference_fallback"):
    display(Markdown("> **Reference fallback:** the generated `hardware_validation.json` was not found."))

if hardware_obs is not None and {"simulator", "hardware"}.issubset(hardware_obs.columns):
    fig, ax = plt.subplots(figsize=(6.5, 6))
    ax.scatter(hardware_obs["simulator"], hardware_obs["hardware"], alpha=0.75)
    low = min(hardware_obs["simulator"].min(), hardware_obs["hardware"].min())
    high = max(hardware_obs["simulator"].max(), hardware_obs["hardware"].max())
    ax.plot([low, high], [low, high], linestyle="--")
    ax.set_xlabel("State-vector feature")
    ax.set_ylabel("IBM hardware feature")
    ax.set_title("Feature-level IBM agreement")
    fig.tight_layout()
    fig.savefig(DASHBOARD / "11_hardware_agreement.png", dpi=180)
    plt.show()
else:
    values = pd.DataFrame({
        "Metric": ["Correlation", "MAE"],
        "Value": [hardware_summary.get("feature_correlation"), hardware_summary.get("feature_mae")],
    })
    fig, ax = plt.subplots(figsize=(6.5, 4))
    bars = ax.bar(values["Metric"], values["Value"])
    ax.set_title("IBM feature-level agreement summary")
    for bar, value in zip(bars, values["Value"]):
        ax.text(bar.get_x() + bar.get_width()/2, value, f"{value:.4f}", ha="center", va="bottom")
    fig.tight_layout()
    fig.savefig(DASHBOARD / "11_hardware_summary.png", dpi=180)
    plt.show()

,Measure,Value
0,Backend,ibm_marrakesh
1,Qubits,7
2,Transpiled depth,71
3,Shots/circuit,4096
4,Validation windows,3
5,Observables,39
6,Feature correlation,0.996332
7,Feature MAE,0.030355


## 11. Honest findings and limitations

**Supported findings**

- Calibrated QRC leads the headline RMSE and QLIKE comparison.
- QRC features support strong calm/turbulent regime detection.
- QRC is competitive with HAR-RV on authentic five-minute realized variance.
- Hardware-native Pauli features show strong agreement between state-vector simulation and IBM hardware.
- Shot, noise, ablation, and scaling studies characterize robustness rather than claiming quantum supremacy.

**Limitations**

- The headline advantage is modest.
- Mincer–Zarnowitz slopes indicate residual bias.
- Authentic-RV performance is a statistical tie with HAR-RV.
- Entanglement-spectrum features did not improve the selected model.
- The tested reservoir-size range does not demonstrate a size advantage.
- Hardware validation is feature-level and targeted, not a full 750-day forecast executed on a QPU.

## 12. Export dashboard summary

In [13]:
summary = {
    "generated_from_submitted_files": bool((provenance_table["Source"] == "submitted generated file").any()),
    "reference_fallbacks_used": provenance_table.loc[
        provenance_table["Source"] == "LOCKED REFERENCE FALLBACK", "Evidence"
    ].tolist(),
    "headline_best_rmse_model": str(headline["RMSE"].idxmin()),
    "headline_best_qlike_model": str(headline["QLIKE"].idxmin()),
    "oxford_best_rmse_model": str(oxford.loc[oxford["RMSE"].idxmin(), "Model"]),
    "hardware_backend": hardware_summary.get("backend"),
    "hardware_feature_correlation": hardware_summary.get("feature_correlation"),
    "hardware_feature_mae": hardware_summary.get("feature_mae"),
    "dashboard_figures": [str(p) for p in sorted(DASHBOARD.glob("*.png"))],
}
(RESULTS / "dashboard_summary.json").write_text(json.dumps(summary, indent=2))
provenance_table.to_csv(RESULTS / "dashboard_provenance.csv", index=False)
print(json.dumps(summary, indent=2))
print("\nSaved dashboard figures:")
for path in sorted(DASHBOARD.glob("*.png")):
    print(" ", path)

{
  "generated_from_submitted_files": true,
  "reference_fallbacks_used": [],
  "headline_best_rmse_model": "QRC Dual+Pauli+FB (calibrated)",
  "headline_best_qlike_model": "QRC Dual+Pauli+FB (calibrated)",
  "oxford_best_rmse_model": "QRC Dual+Pauli+FB",
  "hardware_backend": "ibm_marrakesh",
  "hardware_feature_correlation": 0.9963317843995501,
  "hardware_feature_mae": 0.030355188364938883,
  "dashboard_figures": [
    "dashboard_figures/01_headline_rmse.png",
    "dashboard_figures/02_headline_qlike.png",
    "dashboard_figures/03_regime_detection.png",
    "dashboard_figures/04_oxford_rmse.png",
    "dashboard_figures/05_oxford_qlike.png",
    "dashboard_figures/06_ablation.png",
    "dashboard_figures/07_scaling.png",
    "dashboard_figures/08_encoding_density.png",
    "dashboard_figures/09_shot_budget.png",
    "dashboard_figures/10_noise_zne.png",
    "dashboard_figures/11_hardware_agreement.png"
  ]
}

Saved dashboard figures:
  dashboard_figures/01_headline_rmse.png
  dashbo

In [14]:
import matplotlib.pyplot as plt
plt.close("all")

print("Notebook 2 completed.")
print("Provenance rows:", len(provenance_table))
print("Dashboard figures:", len(list(DASHBOARD.glob("*.png"))))

Notebook 2 completed.
Provenance rows: 9
Dashboard figures: 11


## 13. Transition-event performance


In [15]:
transition_path = Path("results/transition_metrics.csv")
if transition_path.exists():
    transition = pd.read_csv(transition_path)
    display(transition)
    fig, ax = plt.subplots(figsize=(6.8, 3.8))
    x = np.arange(len(transition))
    ax.bar(x - 0.18, transition["qrc_window_accuracy"], width=0.36, label="QRC")
    ax.bar(x + 0.18, transition["majority_window_accuracy"], width=0.36, label="Majority")
    ax.set_xticks(x)
    ax.set_xticklabels([f"+/-{value} days" for value in transition["tolerance_days"]])
    ax.set_ylim(0, 1)
    ax.set_ylabel("Accuracy in transition windows")
    ax.set_title("Transition-focused calm/turbulent evaluation")
    ax.legend()
    fig.tight_layout()
    fig.savefig("dashboard_figures/12_transition_metrics.png", dpi=180)
    plt.show()
else:
    print("Transition metrics are unavailable. Run Notebook 1.")


,tolerance_days,true_transition_events,predicted_transition_events,event_precision,event_recall,event_f1,transition_window_days,qrc_window_accuracy,qrc_window_balanced_accuracy,qrc_window_macro_f1,majority_window_accuracy
0,1,212,64,0.921875,0.377358,0.535512,391,0.595908,0.598036,0.594505,0.514066
1,3,212,64,1.000000,0.547170,0.707317,525,0.687619,0.690226,0.687618,0.531429


## 14. Forecast significance


In [16]:
significance_path = Path("results/forecast_significance.csv")
if significance_path.exists():
    significance = pd.read_csv(significance_path)
    display(significance)
else:
    print("Forecast significance is unavailable. Run Notebook 1.")


,comparison,analysis,metric,estimate,ci_low,ci_high,statistic,p_value,detail
0,QRC calibrated vs HAR-RV,Diebold-Mariano,squared_error,-7.345860e-10,NaN,NaN,-2.993896,2.754398e-03,Newey-West lag 9; negative loss differential favors QRC
1,QRC calibrated vs HAR-RV,Diebold-Mariano,qlike,-8.681826e-02,NaN,NaN,-2.935728,3.327656e-03,Newey-West lag 9; negative loss differential favors QRC
2,QRC calibrated vs HAR-RV,Moving-block bootstrap,rmse,-4.245080e-06,-0.000006,-0.000002,NaN,NaN,95% interval; block length 22; negative favors QRC
3,QRC calibrated vs HAR-RV,Moving-block bootstrap,qlike,-6.873563e-02,-0.110547,-0.030791,NaN,NaN,95% interval; block length 22; negative favors QRC
4,QRC calibrated vs ESN,Diebold-Mariano,squared_error,-7.163234e-10,NaN,NaN,-3.483574,4.947656e-04,Newey-West lag 9; negative loss differential favors QRC
5,QRC calibrated vs ESN,Diebold-Mariano,qlike,-1.327400e-01,NaN,NaN,-4.766742,1.872292e-06,Newey-West lag 9; negative loss differential favors QRC
6,QRC calibrated vs ESN,Moving-block bootstrap,rmse,-4.882254e-06,-0.000007,-0.000002,NaN,NaN,95% interval; block length 22; negative favors QRC
7,QRC calibrated vs ESN,Moving-block bootstrap,qlike,-1.228101e-01,-0.167734,-0.074033,NaN,NaN,95% interval; block length 22; negative favors QRC
8,QRC calibrated vs LSTM,Diebold-Mariano,squared_error,-1.101234e-09,NaN,NaN,-2.693074,7.079659e-03,Newey-West lag 9; negative loss differential favors QRC
9,QRC calibrated vs LSTM,Diebold-Mariano,qlike,-1.036497e-01,NaN,NaN,-3.545691,3.915854e-04,Newey-West lag 9; negative loss differential favors QRC


## 15. Amplitude-damping robustness


In [17]:
amplitude_path = Path("results/amplitude_damping.csv")
if amplitude_path.exists():
    amplitude = pd.read_csv(amplitude_path)
    display(amplitude)
    fig, ax = plt.subplots(figsize=(6.6, 3.8))
    ax.plot(amplitude["amplitude_gamma"], amplitude["feature_mae"], marker="o")
    ax.set_xlabel("Amplitude damping gamma")
    ax.set_ylabel("Feature MAE vs ideal")
    ax.set_title("Combined depolarizing + amplitude-damping noise")
    fig.tight_layout()
    fig.savefig("dashboard_figures/13_amplitude_damping.png", dpi=180)
    plt.show()
else:
    print("Amplitude-damping results are unavailable. Run Notebook 1.")


,amplitude_gamma,depol_1q,depol_2q,feature_mae,windows,qubits
0,0.000,0.005,0.01,0.026138,8,5
1,0.001,0.005,0.01,0.027054,8,5
2,0.005,0.005,0.01,0.031035,8,5
3,0.010,0.005,0.01,0.036178,8,5
4,0.020,0.005,0.01,0.046951,8,5


## 16. Hardware observable scatter


In [18]:
hardware_observables_path = Path("results/hardware_observables.csv")
if hardware_observables_path.exists():
    hardware_points = pd.read_csv(hardware_observables_path)
    display(hardware_points.head())
    fig, ax = plt.subplots(figsize=(5.4, 5.0))
    ax.scatter(hardware_points["simulator"], hardware_points["hardware"], alpha=0.8)
    low = float(min(hardware_points["simulator"].min(), hardware_points["hardware"].min()))
    high = float(max(hardware_points["simulator"].max(), hardware_points["hardware"].max()))
    ax.plot([low, high], [low, high], linestyle="--", label="ideal agreement")
    ax.set_xlabel("Simulator expectation")
    ax.set_ylabel("IBM expectation")
    ax.set_title("IBM Marrakesh: 39 observable pairs")
    ax.legend()
    fig.tight_layout()
    fig.savefig("dashboard_figures/14_hardware_scatter.png", dpi=180)
    plt.show()
else:
    print("Hardware observables are unavailable.")


,backend,validation_role,window,feature_index,job_id,observable_index,simulator,hardware,absolute_error
0,ibm_marrakesh,independent replication and artifact recovery,1,3000,d9cpek9htsac739c2l80,0,-0.574565,-0.535783,0.038782
1,ibm_marrakesh,independent replication and artifact recovery,1,3000,d9cpek9htsac739c2l80,1,-0.131539,-0.099235,0.032304
2,ibm_marrakesh,independent replication and artifact recovery,1,3000,d9cpek9htsac739c2l80,2,-0.189320,-0.216367,0.027047
3,ibm_marrakesh,independent replication and artifact recovery,1,3000,d9cpek9htsac739c2l80,3,-0.520540,-0.512666,0.007873
4,ibm_marrakesh,independent replication and artifact recovery,1,3000,d9cpek9htsac739c2l80,4,-0.653717,-0.619833,0.033884


## 17. Mandatory MNIST benchmark


In [20]:
mnist_path = Path("results/mnist_qrc_metrics.csv")
if mnist_path.exists():
    mnist = pd.read_csv(mnist_path)
    display(mnist)
    fig, ax = plt.subplots(figsize=(6.8, 3.8))
    labels = mnist["qubits"].map(lambda value: "Control" if value == 0 else f"QRC {int(value)}q")
    ax.bar(labels, mnist["accuracy"])
    ax.set_ylim(0, 1)
    ax.set_ylabel("MNIST accuracy")
    ax.set_title("Common QRC benchmark")
    fig.tight_layout()
    fig.savefig("dashboard_figures/15_mnist_accuracy.png", dpi=180)
    plt.show()
else:
    print("MANDATORY OUTPUT MISSING: run Notebook 4 on canonical MNIST. A digits smoke-test file is not accepted.")


,dataset,qubits,reservoir_features,train_samples,test_samples,accuracy,balanced_accuracy,macro_f1,feature_wall_seconds,readout_wall_seconds,seed
0,MNIST,5,18,100,50,0.32,0.32,0.307006,0.216001,0.009107,7
1,MNIST,10,38,100,50,0.60,0.60,0.588709,0.510951,0.016385,7
2,MNIST,15,58,100,50,0.58,0.58,0.567164,1.037265,0.044916,7
3,MNIST,0,16,100,50,0.80,0.80,0.788510,0.000000,0.011345,7
